# Notebook 03 — Experiments 1 and 2: Models and Feature Sets (RQ1)

**Project:** A Machine-Learning Pipeline for Network Intrusion Detection
  |  **Depends on:** Notebooks 01 and 02



## 1. Setup

This cell rebuilds the project paths and imports everything the notebook needs.
A GPU is used for the neural model when one is available.
The random seed is the same one used by the two earlier notebooks.
Run this once at the start of every session.

In [2]:
# --- Environment -----------------------------------------------------------
import os, sys, json, gc, time, math, hashlib, warnings, itertools
from pathlib import Path
from datetime import datetime, timezone

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/nids_benchmark")
else:
    PROJECT = Path.home() / "nids_benchmark"

PROCESSED = PROJECT / "data" / "processed"
ARTIFACTS = PROJECT / "artifacts"
MODELS    = PROJECT / "models" / "nb03"
FIGURES   = PROJECT / "figures" / "nb03"
REPORTS   = PROJECT / "reports" / "nb03"
for p in [MODELS, FIGURES, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn

SEED = 42
random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100); pd.set_option("display.width", 220)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200,
                     "savefig.bbox": "tight", "font.size": 9})

try:
    import torch
    HAS_TORCH = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    HAS_TORCH, DEVICE = False, "cpu"

print("Project :", PROJECT)
print("sklearn :", sklearn.__version__)
print("torch   :", (torch.__version__ + f" on {DEVICE}") if HAS_TORCH else "not available")
if HAS_TORCH and DEVICE == "cuda":
    print("GPU     :", torch.cuda.get_device_name(0))
else:
    print("NOTE: no GPU. Runtime > Change runtime type > T4 GPU speeds the MLP up ~10x.")


Mounted at /content/drive
Project : /content/drive/MyDrive/nids_benchmark
sklearn : 1.6.1
torch   : 2.11.0+cu128 on cuda
GPU     : Tesla T4


This cell redefines the two custom transformer pieces from Notebook 02.
Joblib stores them by name so they must exist before any preprocessor is loaded.
The definitions are byte identical to Notebook 02 including the names patch.
Omitting this cell makes every joblib load fail with an attribute error.

In [3]:
# --- Custom transformer definitions required by joblib ---------------------
from sklearn.base import BaseEstimator, TransformerMixin

class Winsoriser(BaseEstimator, TransformerMixin):
    """Clip each column to a quantile range learned on the training data only."""
    def __init__(self, upper=0.9999, lower=0.0):
        self.upper, self.lower = upper, lower
    def fit(self, X, y=None):
        A = np.asarray(X, dtype="float64")
        self.hi_ = np.nanquantile(A, self.upper, axis=0)
        self.lo_ = np.nanquantile(A, self.lower, axis=0)
        self.n_features_in_ = A.shape[1]
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, dtype="float64"), self.lo_, self.hi_)
    def get_feature_names_out(self, input_features=None):
        if input_features is not None:
            return np.asarray(input_features, dtype=object)
        if hasattr(self, "feature_names_in_"):
            return np.asarray(self.feature_names_in_, dtype=object)
        return np.asarray([f"x{i}" for i in range(getattr(self, "n_features_in_", 0))],
                          dtype=object)

def signed_log1p(A):
    A = np.asarray(A, dtype="float64")
    return np.sign(A) * np.log1p(np.abs(A))

print("Winsoriser and signed_log1p defined in __main__ for joblib compatibility.")


Winsoriser and signed_log1p defined in __main__ for joblib compatibility.


This cell holds every experiment level decision in one editable place.
Set the stage two flag to false for a quick first pass through the notebook.
The row caps exist so a weaker runtime can still complete the whole pipeline.
Record any change here in your project logbook.

In [4]:
# --- Configuration ---------------------------------------------------------
CFG = {
    "primary_dataset": "cic",
    "seeds_stage1": [42, 202, 777],      # variance estimation
    "dev_rows": 400_000,                 # stage 1 sample size
    "run_stage2": True,                  # full-scale final fits
    "stage2_max_rows": None,             # e.g. 800_000 on a weak runtime

    # feature set C
    "k_grid": [10, 15, 20, 25, 30, 40],
    "corr_prune_threshold": 0.98,

    # threshold policy, both chosen on VALIDATION only
    "fpr_budget": 0.01,

    # models
    "lr":  {"max_iter": 500, "C": 1.0},
    "rf":  {"n_estimators": 150, "max_depth": 25, "min_samples_leaf": 5,
            "max_features": "sqrt"},
    "mlp": {"hidden": (256, 128, 64), "dropout": 0.25, "lr": 1e-3,
            "epochs": 60, "batch": 8192, "patience": 7},
}
print(json.dumps({k: str(v) for k, v in CFG.items()}, indent=2))

M = {"notebook": "03_experiments_rq1", "seed": SEED,
     "run_started_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
     "device": DEVICE, "config": CFG, "results": {}}


{
  "primary_dataset": "cic",
  "seeds_stage1": "[42, 202, 777]",
  "dev_rows": "400000",
  "run_stage2": "True",
  "stage2_max_rows": "None",
  "k_grid": "[10, 15, 20, 25, 30, 40]",
  "corr_prune_threshold": "0.98",
  "fpr_budget": "0.01",
  "lr": "{'max_iter': 500, 'C': 1.0}",
  "rf": "{'n_estimators': 150, 'max_depth': 25, 'min_samples_leaf': 5, 'max_features': 'sqrt'}",
  "mlp": "{'hidden': (256, 128, 64), 'dropout': 0.25, 'lr': 0.001, 'epochs': 60, 'batch': 8192, 'patience': 7}"
}


## 2. Reconciliation — rebuilding the state Notebook 02 actually verified

In the Notebook 02 run, the save cell executed at count 25 and the second-pass de-duplication
patch executed at count 27. The parquet files and `splits.json` therefore describe the state
**before** 272,595 further rows were removed and the data re-split. The `.joblib` preprocessors
*were* re-saved inside the patch, so they are correct.

Rather than force a full re-run of Notebook 02, this section reproduces the patch
deterministically — the same hash-based de-duplication on the same modelling columns, the same
split with the same seed — and then **proves** the reconstruction by checking it against the
`nids_fit_rows_` and `nids_fit_hash_` stamps embedded in the saved preprocessor. If those match,
the in-memory state is byte-for-byte the state that passed all 30 hard checks.

If they do not match, the notebook stops. A silent mismatch here would invalidate every result
downstream.

This cell loads the processed frames and the fitted preprocessors from disk.
The preprocessors carry a stamp recording exactly which rows they were fitted on.
That stamp is what makes the reconstruction in the next cell verifiable.
Loading is memory heavy so nothing unnecessary is kept.

In [5]:
# --- Load artefacts ---------------------------------------------------------
FS = json.loads((PROCESSED / "feature_sets.json").read_text())
FSA, CATS = FS["A"], FS["A_categorical"]
FSB_NUM, FSB_CAT = FS["B_numeric"], FS["B_categorical"]

FRAMES, FSB_FRAMES, PREPROC = {}, {}, {}
for ds in ["cic", "unsw", "nsl"]:
    f = PROCESSED / f"{ds}_setA.parquet"
    if f.exists():
        FRAMES[ds] = pd.read_parquet(f)
        print(f"  {ds}_setA.parquet   {FRAMES[ds].shape}")
for ds in ["cic", "unsw"]:
    f = PROCESSED / f"{ds}_setB.parquet"
    if f.exists():
        FSB_FRAMES[ds] = pd.read_parquet(f)
        print(f"  {ds}_setB.parquet   {FSB_FRAMES[ds].shape}")
for ds in ["cic", "unsw", "nsl"]:
    for fs in ["A", "B"]:
        p = ARTIFACTS / f"preproc_{ds}_{fs}.joblib"
        if p.exists():
            PREPROC[(ds, fs)] = joblib.load(p)
print("\npreprocessors loaded:", sorted(f"{d}/{f}" for d, f in PREPROC))
print("stamped fit rows     :",
      {f"{d}/{f}": getattr(p, "nids_fit_rows_", None) for (d, f), p in PREPROC.items()})


  cic_setA.parquet   (2516119, 67)
  unsw_setA.parquet   (2017170, 45)
  nsl_setA.parquet   (147692, 44)
  cic_setB.parquet   (2516119, 23)
  unsw_setB.parquet   (2017170, 23)

preprocessors loaded: ['cic/A', 'cic/B', 'nsl/A', 'unsw/A', 'unsw/B']
stamped fit rows     : {'cic/A': 1435855, 'cic/B': 1435855, 'unsw/A': 1290632, 'unsw/B': 1290632, 'nsl/A': 100240}


This cell reproduces the Notebook 02 patch and then proves the result is exact.
The de-duplication key and the split seed are identical to the patch that was verified.
Success is confirmed by matching the fit hash stored inside the saved preprocessor.
The notebook stops here if the reconstruction cannot be proven correct.

In [6]:
# --- Deterministic reconstruction of the post-patch state -------------------
from sklearn.model_selection import train_test_split

def dedup_model_view(df, cols):
    h = pd.util.hash_pandas_object(df[cols], index=False).to_numpy()
    nlab = df.groupby(pd.Series(h).values, sort=False)["y"].transform("nunique").to_numpy()
    keep = ~(nlab > 1)
    hk = pd.Series(h[keep])
    return np.where(keep)[0][(~hk.duplicated()).to_numpy()]

def strat_key(y_fine, min_n=10):
    vc = y_fine.value_counts()
    rare = set(vc[vc < min_n].index)
    return y_fine.where(~y_fine.isin(rare), "RARE"), sorted(rare)

def make_split(df, test_size=0.20, val_size=0.20, seed=SEED):
    key, _ = strat_key(df["y_fine"])
    idx = np.arange(len(df))
    tv, te = train_test_split(idx, test_size=test_size, stratify=key, random_state=seed)
    tr, va = train_test_split(tv, test_size=val_size, stratify=key.iloc[tv],
                              random_state=seed)
    return tr, va, te

SPLITS, RECON = {}, []
for ds in ["cic", "unsw", "nsl"]:
    if ds not in FRAMES:
        continue
    cols = FSA[ds] + CATS[ds]
    n_before = len(FRAMES[ds])
    keep = dedup_model_view(FRAMES[ds], cols)
    FRAMES[ds] = FRAMES[ds].iloc[keep].reset_index(drop=True)
    if ds in FSB_FRAMES:
        FSB_FRAMES[ds] = FSB_FRAMES[ds].iloc[keep].reset_index(drop=True)

    if ds == "nsl":
        trm = (FRAMES[ds]["orig_partition"] == "train").to_numpy()
        ti = np.where(trm)[0]
        k, _ = strat_key(FRAMES[ds]["y_fine"].iloc[ti])
        tr, va = train_test_split(ti, test_size=0.20, stratify=k, random_state=SEED)
        te = np.where(~trm)[0]
    else:
        tr, va, te = make_split(FRAMES[ds])
    SPLITS[ds] = {"train": tr, "val": va, "test": te}

    stamp = getattr(PREPROC.get((ds, "A")), "nids_fit_rows_", None)
    shash = getattr(PREPROC.get((ds, "A")), "nids_fit_hash_", None)
    myhash = hashlib.md5(np.asarray(sorted(tr)).tobytes()).hexdigest()
    ok = (stamp == len(tr)) and (shash == myhash)
    RECON.append({"dataset": ds, "rows_before": n_before, "rows_after": len(FRAMES[ds]),
                  "removed": n_before - len(FRAMES[ds]), "train": len(tr),
                  "stamp": stamp, "hash_match": shash == myhash, "verified": ok})

rec = pd.DataFrame(RECON)
display(rec)
if not rec["verified"].all():
    raise AssertionError(
        "Reconstruction does NOT match the saved preprocessor stamps.\n"
        "Re-run Notebook 02 sections 10 to 12 and re-save, then return here.")
print("\nRECONSTRUCTION VERIFIED against preprocessor fit stamps for all datasets.")
for ds in FRAMES:
    d = FRAMES[ds]
    print(f"  {ds:5s} {len(d):>9,} rows  attack {100*d['y'].mean():6.3f}%  "
          f"({len(SPLITS[ds]['train']):,}/{len(SPLITS[ds]['val']):,}/"
          f"{len(SPLITS[ds]['test']):,})")
gc.collect()


,dataset,rows_before,rows_after,removed,train,stamp,hash_match,verified
0,cic,2516119,2243524,272595,1435855,1435855,True,True
1,unsw,2017170,2016614,556,1290632,1290632,True,True
2,nsl,147692,147692,0,100240,100240,True,True



RECONSTRUCTION VERIFIED against preprocessor fit stamps for all datasets.
  cic   2,243,524 rows  attack 15.122%  (1,435,855/358,964/448,705)
  unsw  2,016,614 rows  attack  3.869%  (1,290,632/322,659/403,323)
  nsl     147,692 rows  attack 47.953%  (100,240/25,060/22,392)


61

This cell builds the transformed design matrices for the primary dataset.
Each matrix is produced once and reused by every model to save time and memory.
Float32 is used because it halves memory with no measurable effect on results.
The class balance of each partition is printed as a final sanity check.

In [7]:
# --- PATCH: diagnose and repair the design-matrix build --------------------
DS = CFG["primary_dataset"]
df, dfB, sp = FRAMES[DS], FSB_FRAMES[DS], SPLITS[DS]

def required_cols(pre):
    """Exact column names the fitted ColumnTransformer expects at transform time."""
    if hasattr(pre, "feature_names_in_"):
        return list(pre.feature_names_in_)
    cols = []
    for _, _, c in pre.transformers_:
        if isinstance(c, (list, tuple)):
            cols += list(c)
    return cols

print("=" * 72 + "\nCOLUMN AUDIT\n" + "=" * 72)
AUDIT = {}
for tag, pre, frame in [("A", PREPROC[(DS, "A")], df),
                        ("B", PREPROC[(DS, "B")], dfB)]:
    need = required_cols(pre)
    miss = [c for c in need if c not in frame.columns]
    AUDIT[tag] = {"need": need, "missing": miss}
    print(f"\nSet {tag}: preprocessor needs {len(need)} columns, "
          f"frame has {frame.shape[1]}")
    print(f"  missing: {miss if miss else 'none'}")
    if miss:
        print(f"  frame columns (first 25): {list(frame.columns)[:25]}")

# ---- repair: the two frames share a row index so columns can be copied ----
REPAIRS = []
if len(df) == len(dfB):
    for c in AUDIT["A"]["missing"]:
        if c in dfB.columns:
            df[c] = dfB[c].values
            REPAIRS.append(f"A.{c} <- setB")
    for c in AUDIT["B"]["missing"]:
        if c in df.columns:
            dfB[c] = df[c].values
            REPAIRS.append(f"B.{c} <- setA")
else:
    print(f"\nWARNING: row counts differ ({len(df):,} vs {len(dfB):,}); "
          "cross-copy repair is unsafe and was skipped.")

# ---- last resort: rebuild service_grp from a port column if one survived --
PORT_SERVICE = {
    20:"ftp",21:"ftp",22:"ssh",23:"telnet",25:"smtp",53:"dns",67:"dhcp",68:"dhcp",
    69:"tftp",80:"http",110:"pop3",111:"rpc",119:"nntp",123:"ntp",135:"smb",
    137:"smb",138:"smb",139:"smb",143:"imap",161:"snmp",179:"bgp",389:"ldap",
    443:"https",445:"smb",465:"smtp",514:"syslog",587:"smtp",993:"imap",995:"pop3",
    1433:"mssql",1521:"oracle",3306:"mysql",3389:"rdp",5060:"sip",5432:"postgres",
    5900:"vnc",6667:"irc",8000:"http",8080:"http",8443:"https"}

def service_group(port):
    p = pd.to_numeric(port, errors="coerce").fillna(-1).astype("int64")
    band = np.select([p < 0, p < 1024, p < 49152],
                     ["unknown", "other_wellknown", "registered"], default="ephemeral")
    return p.map(PORT_SERVICE).fillna(pd.Series(band, index=p.index)).astype(str)

for tag, frame in [("A", df), ("B", dfB)]:
    if "service_grp" in AUDIT[tag]["missing"] and "service_grp" not in frame.columns:
        src = next((c for c in ["Destination Port", "dsport", "dport"]
                    if c in frame.columns), None)
        if src:
            frame["service_grp"] = service_group(frame[src])
            REPAIRS.append(f"{tag}.service_grp <- rebuilt from {src}")

print("\nrepairs applied:", REPAIRS if REPAIRS else "none needed")
still = {t: [c for c in AUDIT[t]["need"] if c not in (df if t == "A" else dfB).columns]
         for t in ["A", "B"]}
if any(still.values()):
    raise KeyError(
        f"Unrecoverable missing columns: {still}\n"
        "The saved parquet does not contain a column the fitted preprocessor requires.\n"
        "Re-run Notebook 02 section 11 (the save cell) after the de-duplication patch,\n"
        "then return here. Nothing downstream can be trusted until this matches.")
print("all required columns present\n")

# ---- build the design matrices -------------------------------------------
def transform_block(pre, frame, idx):
    """Select exactly what the preprocessor was fitted on, in its own order."""
    X = frame.iloc[idx][required_cols(pre)].copy()
    for c in X.columns:
        if X[c].dtype == object or str(X[c].dtype) == "category":
            X[c] = X[c].astype(str)
    return pre.transform(X).astype(np.float32)

DATA = {}
y_all = df["y"].to_numpy().astype(np.int8)

t0 = time.time()
preA = PREPROC[(DS, "A")]
DATA["A"] = {p: transform_block(preA, df, sp[p]) for p in ["train", "val", "test"]}
print(f"Set A built in {time.time()-t0:.0f}s -> {DATA['A']['train'].shape}")

t0 = time.time()
preB = PREPROC[(DS, "B")]
DATA["B"] = {p: transform_block(preB, dfB, sp[p]) for p in ["train", "val", "test"]}
print(f"Set B built in {time.time()-t0:.0f}s -> {DATA['B']['train'].shape}")

Y = {p: y_all[sp[p]] for p in ["train", "val", "test"]}
for p in ["train", "val", "test"]:
    n1 = int(Y[p].sum())
    print(f"  {p:5s} n={len(Y[p]):>9,}  attack={n1:>8,} ({100*n1/len(Y[p]):.3f}%)")

def names_of(pre, num_cols, cat_cols, width):
    try:
        n = list(pre.get_feature_names_out())
        if len(n) == width:
            return n
    except Exception:
        pass
    n = list(num_cols)
    if cat_cols and "cat" in dict(pre.named_transformers_):
        n += list(pre.named_transformers_["cat"].get_feature_names_out(cat_cols))
    return n if len(n) == width else [f"f{i}" for i in range(width)]

NAMES_A = names_of(preA, FSA[DS], CATS[DS], DATA["A"]["train"].shape[1])
NAMES_B = names_of(preB, FSB_NUM, FSB_CAT, DATA["B"]["train"].shape[1])
print(f"\nfeature names resolved: A={len(NAMES_A)}  B={len(NAMES_B)}")
gc.collect()

COLUMN AUDIT

Set A: preprocessor needs 63 columns, frame has 67
  missing: none

Set B: preprocessor needs 20 columns, frame has 23
  missing: none

repairs applied: none needed
all required columns present

Set A built in 7s -> (1435855, 75)
Set B built in 3s -> (1435855, 32)
  train n=1,435,855  attack= 217,130 (15.122%)
  val   n=  358,964  attack=  54,283 (15.122%)
  test  n=  448,705  attack=  67,854 (15.122%)

feature names resolved: A=75  B=32


128

## 3. Feature Set C — recomputed on the corrected training partition

The ranking saved by Notebook 02 was computed before the second-pass de-duplication, so the
training sample it used overlaps rows that are now in the **test** partition. Using it to choose
Feature Set C would be a genuine selection leak. It is recomputed here on the corrected
training rows only.

Two additions beyond a plain top-*k*:

**A correlation prune.** Notebook 01 found ten CIC column pairs at r = 1.000, but Notebook 02's
hash-based detection only caught some of them — `Total Length of Bwd Packets` and
`Subflow Bwd Bytes` both survived because `safe_downcast` gave them different dtypes and
therefore different hashes. A greedy prune at |r| > 0.98 removes what the hash missed. That
matters most for Logistic Regression, where perfectly collinear inputs make coefficients
uninterpretable and unstable.

**Choosing *k* on validation.** The cut-off is swept over a grid and selected by validation
PR-AUC, never on test.

This cell ranks the transformed features using only the corrected training rows.
Mutual information is univariate while the forest ranking captures interactions.
Both are combined into a mean rank so neither method dominates the choice.
The stale ranking from Notebook 02 is deliberately ignored.

In [8]:
# --- Recompute the ranking on the corrected training partition -------------
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

rng = np.random.default_rng(SEED)
n_rank = min(300_000, len(Y["train"]))
sub = rng.choice(len(Y["train"]), size=n_rank, replace=False)
Xr, yr = DATA["A"]["train"][sub], Y["train"][sub]

mi_n = min(50_000, n_rank)
mi_i = rng.choice(n_rank, size=mi_n, replace=False)
t0 = time.time()
mi = mutual_info_classif(Xr[mi_i], yr[mi_i], random_state=SEED, n_neighbors=3)
print(f"mutual information on {mi_n:,} rows in {time.time()-t0:.0f}s")

t0 = time.time()
rf_rank = RandomForestClassifier(n_estimators=120, max_depth=18, n_jobs=-1,
                                 class_weight="balanced_subsample", random_state=SEED)
rf_rank.fit(Xr, yr)
print(f"random forest ranking on {n_rank:,} rows in {time.time()-t0:.0f}s")

RANK = (pd.DataFrame({"feature": NAMES_A, "mutual_info": mi,
                      "rf_importance": rf_rank.feature_importances_})
          .assign(mi_rank=lambda d: d.mutual_info.rank(ascending=False),
                  rf_rank=lambda d: d.rf_importance.rank(ascending=False))
          .assign(mean_rank=lambda d: (d.mi_rank + d.rf_rank) / 2)
          .sort_values("mean_rank").reset_index(drop=True))
RANK.to_csv(REPORTS / "feature_ranking_corrected.csv", index=False)
display(RANK.head(20).round(5))
del Xr, rf_rank; gc.collect()


mutual information on 50,000 rows in 16s
random forest ranking on 300,000 rows in 23s


,feature,mutual_info,rf_importance,mi_rank,rf_rank,mean_rank
0,Bwd Packet Length Mean,0.31762,0.06280,2.0,4.0,3.0
1,Average Packet Size,0.32063,0.06063,1.0,5.0,3.0
2,Packet Length Variance,0.29901,0.07666,5.0,2.0,3.5
3,Packet Length Std,0.29825,0.05246,6.0,6.0,6.0
4,Total Length of Bwd Packets,0.31387,0.03672,4.0,9.0,6.5
5,Subflow Bwd Bytes,0.31501,0.02718,3.0,11.0,7.0
6,Bwd Packet Length Std,0.27523,0.06593,11.0,3.0,7.0
7,Packet Length Mean,0.28560,0.04183,8.0,8.0,8.0
8,Max Packet Length,0.28089,0.04506,9.0,7.0,8.0
9,Bwd Packet Length Max,0.29077,0.03612,7.0,10.0,8.5


127

This cell prunes features that are near duplicates of a higher ranked feature.
Notebook 02 missed several exact aliases because of a dtype sensitive hash.
Collinear inputs destabilise the linear model and add nothing to the forest.
The pruned pairs are printed so the decision is fully auditable.

In [9]:
# --- Greedy correlation prune ----------------------------------------------
THR = CFG["corr_prune_threshold"]
cs = rng.choice(len(Y["train"]), size=min(150_000, len(Y["train"])), replace=False)
Xc = DATA["A"]["train"][cs]
C = np.corrcoef(Xc, rowvar=False)
C = np.nan_to_num(C)

order = [NAMES_A.index(f) for f in RANK["feature"]]
kept, dropped = [], []
for i in order:
    clash = next((j for j in kept if abs(C[i, j]) > THR), None)
    if clash is None:
        kept.append(i)
    else:
        dropped.append({"dropped": NAMES_A[i], "redundant_with": NAMES_A[clash],
                        "r": round(float(C[i, clash]), 5)})

print(f"Correlation prune at |r| > {THR}: {len(NAMES_A)} -> {len(kept)} features")
if dropped:
    display(pd.DataFrame(dropped).head(20))
else:
    print("  nothing pruned")

RANKED_KEPT = [NAMES_A[i] for i in kept]
M["results"]["corr_pruned"] = dropped
del Xc, C; gc.collect()


Correlation prune at |r| > 0.98: 75 -> 58 features


,dropped,redundant_with,r
0,Packet Length Std,Packet Length Variance,0.99983
1,Subflow Bwd Bytes,Total Length of Bwd Packets,1.00000
2,Packet Length Mean,Average Packet Size,0.99781
3,Bwd Packet Length Max,Bwd Packet Length Mean,0.98121
4,Fwd IAT Mean,Fwd IAT Max,0.99643
5,Fwd IAT Total,Fwd IAT Max,0.99710
6,Subflow Fwd Bytes,Total Length of Fwd Packets,1.00000
7,Flow Duration,Flow IAT Max,0.99250
8,Flow IAT Mean,Flow IAT Max,0.98342
9,Fwd Packets/s,Flow Packets/s,0.99721


31

This cell sweeps the feature count and picks the value that maximises validation score.
A small fast model is used because only the relative ordering of k matters here.
The test partition plays no part in this decision at any point.
The chosen k defines Feature Set C for every experiment that follows.

In [10]:
# --- Choose k on validation -------------------------------------------------
from sklearn.metrics import average_precision_score

idx_of = {n: i for i, n in enumerate(NAMES_A)}
probe_tr = rng.choice(len(Y["train"]), size=min(200_000, len(Y["train"])), replace=False)

sweep = []
for k in CFG["k_grid"]:
    if k > len(RANKED_KEPT):
        continue
    cols = [idx_of[f] for f in RANKED_KEPT[:k]]
    m = RandomForestClassifier(n_estimators=80, max_depth=18, n_jobs=-1,
                               min_samples_leaf=5, class_weight="balanced_subsample",
                               random_state=SEED)
    t0 = time.time()
    m.fit(DATA["A"]["train"][np.ix_(probe_tr, cols)], Y["train"][probe_tr])
    ap = average_precision_score(Y["val"], m.predict_proba(DATA["A"]["val"][:, cols])[:, 1])
    sweep.append({"k": k, "val_pr_auc": round(float(ap), 5),
                  "fit_s": round(time.time() - t0, 1)})
    print(f"  k={k:>3}  val PR-AUC={ap:.5f}  ({time.time()-t0:.0f}s)")
    del m; gc.collect()

SW = pd.DataFrame(sweep)
display(SW)
BEST_K = int(SW.loc[SW.val_pr_auc.idxmax(), "k"])
FSC_NAMES = RANKED_KEPT[:BEST_K]
FSC_IDX = [idx_of[f] for f in FSC_NAMES]
DATA["C"] = {p: DATA["A"][p][:, FSC_IDX] for p in ["train", "val", "test"]}

print(f"\nFeature Set C = top {BEST_K} of the pruned ranking, chosen on validation PR-AUC")
print("  " + ", ".join(FSC_NAMES))
M["results"]["feature_set_C"] = {"k": BEST_K, "features": FSC_NAMES,
                                 "sweep": sweep}
NAMES = {"A": NAMES_A, "B": NAMES_B, "C": FSC_NAMES}
for fs in ["A", "B", "C"]:
    print(f"  Set {fs}: {DATA[fs]['train'].shape[1]} transformed columns")


  k= 10  val PR-AUC=0.99901  (5s)
  k= 15  val PR-AUC=0.99982  (5s)
  k= 20  val PR-AUC=0.99979  (7s)
  k= 25  val PR-AUC=0.99978  (8s)
  k= 30  val PR-AUC=0.99988  (7s)
  k= 40  val PR-AUC=0.99988  (8s)


,k,val_pr_auc,fit_s
0,10,0.99901,4.9
1,15,0.99982,5.3
2,20,0.99979,6.6
3,25,0.99978,8.0
4,30,0.99988,7.5
5,40,0.99988,8.2



Feature Set C = top 30 of the pruned ranking, chosen on validation PR-AUC
  Bwd Packet Length Mean, Average Packet Size, Packet Length Variance, Total Length of Bwd Packets, Bwd Packet Length Std, Max Packet Length, service_grp_http, Init_Win_bytes_backward, Fwd IAT Max, Total Length of Fwd Packets, Fwd Packet Length Max, Init_Win_bytes_forward, Fwd IAT Std, Bwd Packets/s, Fwd Header Length, Fwd Packet Length Mean, Bwd Header Length, Total Backward Packets, Flow IAT Std, Flow IAT Max, Fwd Packet Length Std, Flow Packets/s, Bwd Packet Length Min, Total Fwd Packets, Flow Bytes/s, act_data_pkt_fwd, Bwd IAT Max, Fwd IAT Min, Idle Mean, service_grp_https
  Set A: 75 transformed columns
  Set B: 32 transformed columns
  Set C: 30 transformed columns


## 4. Evaluation harness

The proposal (§3.9) requires confusion matrix, precision, recall, F1, FPR, FNR, ROC-AUC,
PR-AUC, balanced accuracy and MCC, with the **attack class as positive**, plus training and
inference cost. All of that is produced by one function so no experiment can accidentally
report a different metric set.

Thresholds are fitted on validation and then applied unchanged to test. Reporting a test score
at a threshold tuned on test is one of the most common silent errors in this literature.

This cell defines the single metric function used by every experiment.
The attack class is the positive class in every metric that depends on that choice.
Both thresholds are chosen on validation and then applied unchanged to test.
Inference cost is measured because a slow detector is not deployable.

In [11]:
# --- Metric harness ---------------------------------------------------------
from sklearn.metrics import (confusion_matrix, roc_auc_score, average_precision_score,
                             balanced_accuracy_score, matthews_corrcoef,
                             precision_recall_curve, roc_curve)

def pick_thresholds(y_val, p_val, fpr_budget=CFG["fpr_budget"]):
    """Choose an F1-optimal and a fixed-FPR threshold using validation data only."""
    prec, rec, thr = precision_recall_curve(y_val, p_val)
    f1 = np.divide(2 * prec * rec, prec + rec,
                   out=np.zeros_like(prec), where=(prec + rec) > 0)
    thr_f1 = float(thr[max(0, np.nanargmax(f1) - 1)]) if len(thr) else 0.5

    fpr, tpr, rthr = roc_curve(y_val, p_val)
    ok = np.where(fpr <= fpr_budget)[0]
    thr_fpr = float(rthr[ok[-1]]) if len(ok) else 1.0
    thr_fpr = min(max(thr_fpr, 0.0), 1.0)
    return {"f1": thr_f1, f"fpr{int(fpr_budget*100)}": thr_fpr}

def at_threshold(y, p, t):
    yh = (p >= t).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec  = tp / (tp + fn) if (tp + fn) else 0.0
    return {"threshold": round(float(t), 6),
            "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
            "precision": round(prec, 5), "recall": round(rec, 5),
            "f1": round(2 * prec * rec / (prec + rec), 5) if (prec + rec) else 0.0,
            "fpr": round(fp / (fp + tn), 6) if (fp + tn) else 0.0,
            "fnr": round(fn / (fn + tp), 6) if (fn + tp) else 0.0,
            "accuracy": round((tp + tn) / len(y), 5),
            "balanced_accuracy": round(balanced_accuracy_score(y, yh), 5),
            "mcc": round(matthews_corrcoef(y, yh), 5)}

def evaluate(model_name, feature_set, y_val, p_val, y_test, p_test,
             fit_seconds, infer_seconds, n_infer, extra=None):
    thr = pick_thresholds(y_val, p_val)
    row = {"model": model_name, "feature_set": feature_set,
           "roc_auc": round(float(roc_auc_score(y_test, p_test)), 5),
           "pr_auc": round(float(average_precision_score(y_test, p_test)), 5),
           "val_pr_auc": round(float(average_precision_score(y_val, p_val)), 5),
           "fit_seconds": round(fit_seconds, 1),
           "infer_us_per_flow": round(1e6 * infer_seconds / max(n_infer, 1), 2)}
    for tag, t in thr.items():
        for k, v in at_threshold(y_test, p_test, t).items():
            row[f"{tag}_{k}"] = v
    if extra:
        row.update(extra)
    return row

print("Metric harness ready.")
print(f"Thresholds: F1-optimal and recall at FPR <= {100*CFG['fpr_budget']:.0f}%")
print("Both are selected on validation and applied unchanged to test.")


Metric harness ready.
Thresholds: F1-optimal and recall at FPR <= 1%
Both are selected on validation and applied unchanged to test.


## 5. Models

Three models, matching proposal §3.7: a transparent linear baseline, a non-linear ensemble
baseline, and a learned-representation model.

Class imbalance is handled by **weighting, not resampling**, in all three — `class_weight`
for the two scikit-learn models and `pos_weight` in the loss for the MLP. The proposal
explicitly rules out naive resampling of the whole set, and weighting has the practical
advantage of leaving the validation and test distributions untouched.

The MLP uses early stopping on **validation PR-AUC** rather than loss. On a 1:5.6 problem the
loss can keep improving while minority-class ranking degrades, so stopping on the metric you
actually care about is worth several points of PR-AUC.

This cell defines a compact multilayer perceptron with class weighted loss.
Early stopping watches validation PR-AUC rather than loss because the task is imbalanced.
Batch normalisation and dropout keep a wide first layer from overfitting.
The model falls back to a scikit-learn equivalent when torch is unavailable.

In [12]:
# --- Neural model -----------------------------------------------------------
if HAS_TORCH:
    import torch.nn as nn
    from torch.utils.data import TensorDataset, DataLoader

    class TorchMLP:
        def __init__(self, in_dim, hidden=(256, 128, 64), dropout=0.25, lr=1e-3,
                     epochs=60, batch=8192, patience=7, seed=SEED, pos_weight=1.0):
            torch.manual_seed(seed); np.random.seed(seed)
            layers, d = [], in_dim
            for h in hidden:
                layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(),
                           nn.Dropout(dropout)]
                d = h
            layers += [nn.Linear(d, 1)]
            self.net = nn.Sequential(*layers).to(DEVICE)
            self.lr, self.epochs, self.batch = lr, epochs, batch
            self.patience, self.pos_weight = patience, pos_weight
            self.history_ = []

        def fit(self, Xtr, ytr, Xva, yva):
            opt = torch.optim.AdamW(self.net.parameters(), lr=self.lr, weight_decay=1e-4)
            sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max",
                                                               factor=0.5, patience=3)
            lossf = nn.BCEWithLogitsLoss(
                pos_weight=torch.tensor([self.pos_weight], device=DEVICE))
            dl = DataLoader(TensorDataset(torch.from_numpy(Xtr),
                                          torch.from_numpy(ytr.astype(np.float32))),
                            batch_size=self.batch, shuffle=True, drop_last=False)
            best, best_state, bad = -1.0, None, 0
            for ep in range(self.epochs):
                self.net.train()
                for xb, yb in dl:
                    xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE)
                    opt.zero_grad()
                    loss = lossf(self.net(xb).squeeze(1), yb)
                    loss.backward(); opt.step()
                ap = average_precision_score(yva, self.predict_proba(Xva))
                self.history_.append({"epoch": ep, "val_pr_auc": float(ap)})
                sched.step(ap)
                if ap > best + 1e-5:
                    best, bad = ap, 0
                    best_state = {k: v.detach().clone()
                                  for k, v in self.net.state_dict().items()}
                else:
                    bad += 1
                    if bad >= self.patience:
                        break
            if best_state:
                self.net.load_state_dict(best_state)
            self.best_val_pr_auc_ = best
            self.epochs_run_ = len(self.history_)
            return self

        @torch.no_grad()
        def predict_proba(self, X, batch=65536):
            self.net.eval()
            out = np.empty(len(X), dtype=np.float32)
            for i in range(0, len(X), batch):
                xb = torch.from_numpy(X[i:i+batch]).to(DEVICE)
                out[i:i+batch] = torch.sigmoid(
                    self.net(xb).squeeze(1)).cpu().numpy()
            return out
    print(f"TorchMLP defined on {DEVICE}")
else:
    from sklearn.neural_network import MLPClassifier
    print("torch unavailable; sklearn MLPClassifier fallback will be used")


TorchMLP defined on cuda


This cell wraps the three models behind one interface used by every experiment.
Each returns fitted predictions plus the time spent training and predicting.
Class weighting is applied consistently so no model gets an unfair advantage.
Random Forest is capped in depth so it cannot memorise its way to a perfect score.

In [13]:
# --- Unified model runners --------------------------------------------------
from sklearn.linear_model import LogisticRegression

def run_lr(Xtr, ytr, Xva, Xte, seed=SEED):
    m = LogisticRegression(max_iter=CFG["lr"]["max_iter"], C=CFG["lr"]["C"],
                           class_weight="balanced", solver="lbfgs",
                           n_jobs=-1, random_state=seed)
    t0 = time.time(); m.fit(Xtr, ytr); fit_s = time.time() - t0
    pv = m.predict_proba(Xva)[:, 1]
    t0 = time.time(); pt = m.predict_proba(Xte)[:, 1]; inf_s = time.time() - t0
    return m, pv, pt, fit_s, inf_s, {"converged": bool(m.n_iter_[0] < CFG["lr"]["max_iter"])}

def run_rf(Xtr, ytr, Xva, Xte, seed=SEED):
    m = RandomForestClassifier(n_estimators=CFG["rf"]["n_estimators"],
                               max_depth=CFG["rf"]["max_depth"],
                               min_samples_leaf=CFG["rf"]["min_samples_leaf"],
                               max_features=CFG["rf"]["max_features"],
                               class_weight="balanced_subsample",
                               n_jobs=-1, random_state=seed)
    t0 = time.time(); m.fit(Xtr, ytr); fit_s = time.time() - t0
    pv = m.predict_proba(Xva)[:, 1]
    t0 = time.time(); pt = m.predict_proba(Xte)[:, 1]; inf_s = time.time() - t0
    return m, pv, pt, fit_s, inf_s, {}

def run_mlp(Xtr, ytr, Xva, yva, Xte, seed=SEED):
    pw = float((ytr == 0).sum() / max((ytr == 1).sum(), 1))
    if HAS_TORCH:
        m = TorchMLP(Xtr.shape[1], hidden=CFG["mlp"]["hidden"],
                     dropout=CFG["mlp"]["dropout"], lr=CFG["mlp"]["lr"],
                     epochs=CFG["mlp"]["epochs"], batch=CFG["mlp"]["batch"],
                     patience=CFG["mlp"]["patience"], seed=seed, pos_weight=pw)
        t0 = time.time(); m.fit(Xtr, ytr, Xva, yva); fit_s = time.time() - t0
        pv = m.predict_proba(Xva)
        t0 = time.time(); pt = m.predict_proba(Xte); inf_s = time.time() - t0
        return m, pv, pt, fit_s, inf_s, {"epochs_run": m.epochs_run_}
    m = MLPClassifier(hidden_layer_sizes=CFG["mlp"]["hidden"], early_stopping=True,
                      n_iter_no_change=CFG["mlp"]["patience"], max_iter=200,
                      random_state=seed)
    t0 = time.time(); m.fit(Xtr, ytr); fit_s = time.time() - t0
    pv = m.predict_proba(Xva)[:, 1]
    t0 = time.time(); pt = m.predict_proba(Xte)[:, 1]; inf_s = time.time() - t0
    return m, pv, pt, fit_s, inf_s, {"epochs_run": int(m.n_iter_)}

MODEL_RUNNERS = {"LogisticRegression": run_lr, "RandomForest": run_rf, "MLP": run_mlp}
print("Runners ready:", list(MODEL_RUNNERS))
print(f"pos_weight for MLP will be ~{(Y['train']==0).sum()/max((Y['train']==1).sum(),1):.2f}")


Runners ready: ['LogisticRegression', 'RandomForest', 'MLP']
pos_weight for MLP will be ~5.61


## 6. Stage 1 — development scale with repeated runs

Nine configurations (3 models × 3 feature sets), each repeated over three seeds on a stratified
development subsample. The point is not the headline number; it is the **spread**. Proposal
§3.8 asks for *"a range of values reported rather than a single value"*, and a difference between
two models is only meaningful if it exceeds the run-to-run variation.

This cell runs every model and feature set combination across three seeds.
The development subsample keeps the whole sweep inside a few minutes.
Reporting a spread rather than one number is what makes model comparison honest.
Progress is printed per configuration because the loop is long.

In [14]:
# --- Stage 1 sweep ----------------------------------------------------------
dev_n = min(CFG["dev_rows"], len(Y["train"]))
dev_rng = np.random.default_rng(SEED)
pos = np.where(Y["train"] == 1)[0]
neg = np.where(Y["train"] == 0)[0]
frac = dev_n / len(Y["train"])
dev_idx = np.sort(np.concatenate([
    dev_rng.choice(pos, int(len(pos) * frac), replace=False),
    dev_rng.choice(neg, int(len(neg) * frac), replace=False)]))
print(f"Development subsample: {len(dev_idx):,} rows  "
      f"attack {100*Y['train'][dev_idx].mean():.3f}% (stratification preserved)")

stage1 = []
for fs in ["A", "B", "C"]:
    Xtr_d = DATA[fs]["train"][dev_idx]
    ytr_d = Y["train"][dev_idx]
    Xva, Xte = DATA[fs]["val"], DATA[fs]["test"]
    for mname in MODEL_RUNNERS:
        for seed in CFG["seeds_stage1"]:
            t0 = time.time()
            if mname == "MLP":
                _, pv, pt, fs_s, in_s, ex = run_mlp(Xtr_d, ytr_d, Xva, Y["val"],
                                                    Xte, seed=seed)
            else:
                _, pv, pt, fs_s, in_s, ex = MODEL_RUNNERS[mname](
                    Xtr_d, ytr_d, Xva, Xte, seed=seed)
            r = evaluate(mname, fs, Y["val"], pv, Y["test"], pt,
                         fs_s, in_s, len(Y["test"]), extra={"seed": seed, **ex})
            r["stage"] = "dev"
            stage1.append(r)
            print(f"  {fs} | {mname:18s} seed={seed:<4} "
                  f"PR-AUC={r['pr_auc']:.4f}  "
                  f"DR@1%FPR={r['fpr1_recall']:.4f}  ({time.time()-t0:.0f}s)")
            gc.collect()

S1 = pd.DataFrame(stage1)
S1.to_csv(REPORTS / "stage1_dev_runs.csv", index=False)
print(f"\nStage 1 complete: {len(S1)} runs")


Development subsample: 399,999 rows  attack 15.122% (stratification preserved)
  A | LogisticRegression seed=42   PR-AUC=0.9940  DR@1%FPR=0.9907  (23s)
  A | LogisticRegression seed=202  PR-AUC=0.9940  DR@1%FPR=0.9907  (23s)
  A | LogisticRegression seed=777  PR-AUC=0.9940  DR@1%FPR=0.9907  (24s)
  A | RandomForest       seed=42   PR-AUC=0.9999  DR@1%FPR=0.9998  (45s)
  A | RandomForest       seed=202  PR-AUC=0.9999  DR@1%FPR=0.9998  (46s)
  A | RandomForest       seed=777  PR-AUC=0.9999  DR@1%FPR=0.9998  (44s)
  A | MLP                seed=42   PR-AUC=0.9999  DR@1%FPR=0.9995  (150s)
  A | MLP                seed=202  PR-AUC=0.9999  DR@1%FPR=0.9995  (113s)
  A | MLP                seed=777  PR-AUC=0.9999  DR@1%FPR=0.9995  (125s)
  B | LogisticRegression seed=42   PR-AUC=0.9454  DR@1%FPR=0.9574  (7s)
  B | LogisticRegression seed=202  PR-AUC=0.9454  DR@1%FPR=0.9574  (7s)
  B | LogisticRegression seed=777  PR-AUC=0.9454  DR@1%FPR=0.9574  (7s)
  B | RandomForest       seed=42   PR-AUC=0.9

OSError: [Errno 5] Input/output error: '/content/drive/MyDrive/nids_benchmark/reports/nb03/stage1_dev_runs.csv'

In [15]:
# Save directly to Colab's local temporary storage
S1 = pd.DataFrame(stage1)
S1.to_csv("/content/stage1_dev_runs.csv", index=False)
print(f"Rescued {len(S1)} runs to /content/stage1_dev_runs.csv!")

Rescued 27 runs to /content/stage1_dev_runs.csv!


In [16]:
import os
from google.colab import drive

# 1. Force remount Google Drive to clear stale connection
drive.mount('/content/drive', force_remount=True)

# 2. Ensure target directory exists
REPORTS.mkdir(parents=True, exist_ok=True)

# 3. Save to Google Drive
output_path = REPORTS / "stage1_dev_runs.csv"
S1.to_csv(output_path, index=False)
print(f"Successfully saved to Google Drive at: {output_path}")

Mounted at /content/drive
Successfully saved to Google Drive at: /content/drive/MyDrive/nids_benchmark/reports/nb03/stage1_dev_runs.csv


This cell summarises the sweep as a mean and standard deviation per configuration.
A difference smaller than the spread is not evidence of a better model.
Detection rate at a one percent false positive budget is the operational headline.
This table is the direct answer to the first half of research question one.

In [17]:
# --- Stage 1 summary --------------------------------------------------------
agg = (S1.groupby(["feature_set", "model"])
         .agg(pr_auc_mean=("pr_auc", "mean"), pr_auc_std=("pr_auc", "std"),
              roc_auc_mean=("roc_auc", "mean"),
              dr_at_1fpr_mean=("fpr1_recall", "mean"),
              dr_at_1fpr_std=("fpr1_recall", "std"),
              f1_mean=("f1_f1", "mean"), f1_std=("f1_f1", "std"),
              fpr_at_f1_mean=("f1_fpr", "mean"),
              mcc_mean=("f1_mcc", "mean"),
              fit_s_mean=("fit_seconds", "mean"),
              infer_us_mean=("infer_us_per_flow", "mean"))
         .round(5).reset_index())
display(agg)
agg.to_csv(REPORTS / "stage1_summary.csv", index=False)

print("\nRanking by detection rate at 1% FPR (mean over 3 seeds):")
top = agg.sort_values("dr_at_1fpr_mean", ascending=False)
for _, r in top.iterrows():
    print(f"  {r.feature_set} / {r.model:18s} "
          f"DR@1%FPR={r.dr_at_1fpr_mean:.4f} ± {r.dr_at_1fpr_std:.4f}   "
          f"PR-AUC={r.pr_auc_mean:.4f} ± {r.pr_auc_std:.4f}")

best = top.iloc[0]
print(f"\nBest development configuration: {best.model} on Feature Set {best.feature_set}")
print(f"Largest seed-to-seed spread observed: "
      f"{agg.pr_auc_std.max():.5f} PR-AUC — differences below this are not meaningful.")
M["results"]["stage1"] = agg.to_dict("records")


,feature_set,model,pr_auc_mean,pr_auc_std,roc_auc_mean,dr_at_1fpr_mean,dr_at_1fpr_std,f1_mean,f1_std,fpr_at_f1_mean,mcc_mean,fit_s_mean,infer_us_mean
0,A,LogisticRegression,0.99402,0.00000,0.99882,0.99067,0.00000,0.97442,0.00000,0.00592,0.96986,22.66667,0.24000
1,A,MLP,0.99986,0.00001,0.99997,0.99949,0.00002,0.99576,0.00015,0.00038,0.99501,128.36667,0.16000
2,A,RandomForest,0.99992,0.00000,0.99998,0.99980,0.00002,0.99696,0.00006,0.00050,0.99642,43.36667,1.83333
3,B,LogisticRegression,0.94537,0.00000,0.99497,0.95736,0.00000,0.95402,0.00000,0.01137,0.94585,5.90000,0.12667
4,B,MLP,0.99952,0.00004,0.99990,0.99803,0.00026,0.99173,0.00016,0.00046,0.99029,172.96667,0.12000
5,B,RandomForest,0.99981,0.00000,0.99996,0.99972,0.00001,0.99377,0.00010,0.00037,0.99268,27.36667,1.40333
6,C,LogisticRegression,0.97830,0.00000,0.99472,0.94106,0.00000,0.94434,0.00000,0.00503,0.93519,6.10000,0.15333
7,C,MLP,0.99972,0.00002,0.99993,0.99892,0.00016,0.99464,0.00017,0.00043,0.99370,144.46667,0.15000
8,C,RandomForest,0.99992,0.00001,0.99998,0.99979,0.00004,0.99700,0.00003,0.00049,0.99646,32.13333,2.02667



Ranking by detection rate at 1% FPR (mean over 3 seeds):
  A / RandomForest       DR@1%FPR=0.9998 ± 0.0000   PR-AUC=0.9999 ± 0.0000
  C / RandomForest       DR@1%FPR=0.9998 ± 0.0000   PR-AUC=0.9999 ± 0.0000
  B / RandomForest       DR@1%FPR=0.9997 ± 0.0000   PR-AUC=0.9998 ± 0.0000
  A / MLP                DR@1%FPR=0.9995 ± 0.0000   PR-AUC=0.9999 ± 0.0000
  C / MLP                DR@1%FPR=0.9989 ± 0.0002   PR-AUC=0.9997 ± 0.0000
  B / MLP                DR@1%FPR=0.9980 ± 0.0003   PR-AUC=0.9995 ± 0.0000
  A / LogisticRegression DR@1%FPR=0.9907 ± 0.0000   PR-AUC=0.9940 ± 0.0000
  B / LogisticRegression DR@1%FPR=0.9574 ± 0.0000   PR-AUC=0.9454 ± 0.0000
  C / LogisticRegression DR@1%FPR=0.9411 ± 0.0000   PR-AUC=0.9783 ± 0.0000

Best development configuration: RandomForest on Feature Set A
Largest seed-to-seed spread observed: 0.00004 PR-AUC — differences below this are not meaningful.


## 7. Stage 2 — full training partition

The development results establish the ordering and the noise floor. Stage 2 refits every
configuration once on the full 1.43M-row training partition to produce the headline table for
the dissertation. Only the seed count changes; nothing else about the protocol moves, so the
two stages are directly comparable and the difference between them is itself informative about
how much data these models actually need.

This cell refits every configuration once on the full training partition.
Only the amount of data changes so the two stages remain directly comparable.
Random Forest dominates the runtime here so expect a long wait.
Set the stage two flag to false in the configuration to skip this step.

In [ ]:
# Explicitly enforce directory creation on disk before running models
MODELS.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

# --- Stage 2 full-scale runs ------------------------------------------------
stage2 = []
MLP_HISTORY, RF_IMPORTANCE, LR_COEF = {}, {}, {}

if CFG["run_stage2"]:
    cap = CFG["stage2_max_rows"]
    full_idx = np.arange(len(Y["train"])) if cap is None else dev_rng.choice(
        len(Y["train"]), size=min(cap, len(Y["train"])), replace=False)
    print(f"Stage 2 training rows: {len(full_idx):,}")

    for fs in ["A", "B", "C"]:
        Xtr_f, ytr_f = DATA[fs]["train"][full_idx], Y["train"][full_idx]
        Xva, Xte = DATA[fs]["val"], DATA[fs]["test"]
        for mname in MODEL_RUNNERS:
            t0 = time.time()
            if mname == "MLP":
                mdl, pv, pt, f_s, i_s, ex = run_mlp(Xtr_f, ytr_f, Xva, Y["val"], Xte)
                if HAS_TORCH:
                    MLP_HISTORY[fs] = mdl.history_
            else:
                mdl, pv, pt, f_s, i_s, ex = MODEL_RUNNERS[mname](Xtr_f, ytr_f, Xva, Xte)
                if mname == "RandomForest":
                    RF_IMPORTANCE[fs] = mdl.feature_importances_.copy()
                if mname == "LogisticRegression":
                    LR_COEF[fs] = mdl.coef_[0].copy()
            r = evaluate(mname, fs, Y["val"], pv, Y["test"], pt,
                         f_s, i_s, len(Y["test"]), extra={"seed": SEED, **ex})
            r["stage"] = "full"
            stage2.append(r)

            # Defensive directory creation inside loop to handle Drive drops
            REPORTS.mkdir(parents=True, exist_ok=True)
            MODELS.mkdir(parents=True, exist_ok=True)

            np.save(REPORTS / f"probs_full_{mname}_{fs}.npy", pt.astype(np.float32))
            if mname != "MLP":
                joblib.dump(mdl, MODELS / f"{mname}_{fs}_full.joblib", compress=3)
            print(f"  {fs} | {mname:18s} PR-AUC={r['pr_auc']:.4f}  "
                  f"DR@1%FPR={r['fpr1_recall']:.4f}  F1={r['f1_f1']:.4f}  "
                  f"({time.time()-t0:.0f}s)")
            del mdl; gc.collect()
    S2 = pd.DataFrame(stage2)
    S2.to_csv(REPORTS / "stage2_full_runs.csv", index=False)
else:
    S2 = pd.DataFrame()
    print("Stage 2 skipped (CFG['run_stage2'] is False).")
print(f"\nStage 2 complete: {len(S2)} runs")

Stage 2 training rows: 1,435,855
  A | LogisticRegression PR-AUC=0.9937  DR@1%FPR=0.9910  F1=0.9743  (81s)
  A | RandomForest       PR-AUC=1.0000  DR@1%FPR=0.9999  F1=0.9975  (197s)
  A | MLP                PR-AUC=0.9999  DR@1%FPR=0.9997  F1=0.9964  (353s)
  B | LogisticRegression PR-AUC=0.9448  DR@1%FPR=0.9563  F1=0.9532  (21s)
  B | RandomForest       PR-AUC=0.9999  DR@1%FPR=0.9999  F1=0.9948  (127s)
  B | MLP                PR-AUC=0.9997  DR@1%FPR=0.9992  F1=0.9931  (556s)
  C | LogisticRegression PR-AUC=0.9781  DR@1%FPR=0.9416  F1=0.9434  (20s)
  C | RandomForest       PR-AUC=1.0000  DR@1%FPR=0.9999  F1=0.9979  (140s)


This cell prints the headline results table for research question one.
Detection rate at a fixed false positive budget is shown next to the threshold free scores.
The false alert count per million flows makes the operational cost concrete.
This table goes straight into the results chapter of the dissertation.

In [ ]:
# --- Headline RQ1 table -----------------------------------------------------
RES = S2 if len(S2) else S1[S1.seed == SEED]
cols = ["feature_set", "model", "roc_auc", "pr_auc",
        "fpr1_recall", "fpr1_precision", "fpr1_fpr",
        "f1_f1", "f1_recall", "f1_precision", "f1_fpr", "f1_mcc",
        "f1_balanced_accuracy", "fit_seconds", "infer_us_per_flow"]
head = RES[cols].copy()
head["false_alerts_per_1M_flows"] = (head["f1_fpr"] * 1e6).round(0).astype(int)
head = head.sort_values(["feature_set", "pr_auc"], ascending=[True, False])
display(head.round(5))
head.to_csv(REPORTS / "rq1_headline.csv", index=False)

print("\nDegradation from Feature Set A (PR-AUC):")
base = head[head.feature_set == "A"].set_index("model")["pr_auc"]
for fs in ["B", "C"]:
    sub = head[head.feature_set == fs].set_index("model")["pr_auc"]
    for m in base.index:
        if m in sub.index:
            d = sub[m] - base[m]
            print(f"  {m:18s} A -> {fs}: {base[m]:.4f} -> {sub[m]:.4f} "
                  f"({100*d/base[m]:+.2f}%)")

nA = DATA["A"]["train"].shape[1]; nC = DATA["C"]["train"].shape[1]
print(f"\nFeature Set C uses {nC} of {nA} columns ({100*nC/nA:.0f}%). "
      f"Judge the accuracy cost against that reduction.")
M["results"]["rq1_headline"] = head.to_dict("records")


## 8. The UNSW artefact ablation

Notebook 01 found `sttl` at AUC 0.9905 and `ct_state_ttl` at 0.9912 on UNSW-NB15 — a single
integer nearly solving the dataset, because the IXIA generator emitted a near-fixed TTL for
attack traffic. Notebook 02 excluded all three TTL-derived features from Feature Set B but kept
them in UNSW's Feature Set A precisely so the effect could be measured here.

This is the most defensible result in the whole project: it puts a number on how much of a
published-looking score comes from a testbed artefact rather than from learnable signal, and it
explains in advance why cross-dataset transfer in Notebook 04 will not reproduce it.

This cell measures how much of the UNSW score comes from three testbed artefacts.
The same model is trained with and without the TTL derived features.
The gap is the amount of apparent performance that will never transfer.
UNSW is now severely imbalanced so the fixed budget metric matters most here.

In [ ]:
# --- UNSW sttl / ct_state_ttl ablation --------------------------------------
ABL = []
if "unsw" in FRAMES:
    du, spu = FRAMES["unsw"], SPLITS["unsw"]
    preU = PREPROC[("unsw", "A")]
    yu = du["y"].to_numpy().astype(np.int8)
    Yu = {p: yu[spu[p]] for p in ["train", "val", "test"]}

    Xu = {p: transform_block(preU, du, FSA["unsw"], CATS["unsw"], spu[p])
          for p in ["train", "val", "test"]}
    NU = names_of(preU, FSA["unsw"], CATS["unsw"], Xu["train"].shape[1])
    art = [i for i, n in enumerate(NU) if n in FS.get("unsw_artefacts",
                                                      ["sttl", "dttl", "ct_state_ttl"])]
    keep_i = [i for i in range(len(NU)) if i not in art]
    print(f"UNSW: {len(Yu['train']):,} train rows, attack {100*Yu['train'].mean():.3f}%")
    print(f"Artefact columns removed in the ablation: {[NU[i] for i in art]}")

    ab_n = min(300_000, len(Yu["train"]))
    ab_i = np.sort(np.random.default_rng(SEED).choice(len(Yu["train"]), ab_n,
                                                      replace=False))
    for tag, cols in [("with_artefacts", list(range(len(NU)))),
                      ("without_artefacts", keep_i)]:
        for mname in ["RandomForest", "LogisticRegression"]:
            runner = MODEL_RUNNERS[mname]
            _, pv, pt, f_s, i_s, _ = runner(Xu["train"][np.ix_(ab_i, cols)],
                                            Yu["train"][ab_i],
                                            Xu["val"][:, cols], Xu["test"][:, cols])
            r = evaluate(mname, tag, Yu["val"], pv, Yu["test"], pt,
                         f_s, i_s, len(Yu["test"]))
            ABL.append(r)
            print(f"  {mname:18s} {tag:18s} PR-AUC={r['pr_auc']:.4f}  "
                  f"ROC-AUC={r['roc_auc']:.4f}  DR@1%FPR={r['fpr1_recall']:.4f}")
            gc.collect()

    A = pd.DataFrame(ABL)
    display(A[["model", "feature_set", "roc_auc", "pr_auc",
               "fpr1_recall", "f1_f1"]].round(5))
    A.to_csv(REPORTS / "unsw_artefact_ablation.csv", index=False)
    print("\nCost of removing three TTL-derived columns:")
    for m in A.model.unique():
        w  = A[(A.model == m) & (A.feature_set == "with_artefacts")].iloc[0]
        wo = A[(A.model == m) & (A.feature_set == "without_artefacts")].iloc[0]
        print(f"  {m:18s} PR-AUC {w.pr_auc:.4f} -> {wo.pr_auc:.4f} "
              f"({100*(wo.pr_auc-w.pr_auc)/w.pr_auc:+.1f}%)   "
              f"ROC-AUC {w.roc_auc:.4f} -> {wo.roc_auc:.4f}")
    M["results"]["unsw_ablation"] = A.to_dict("records")
    del Xu; gc.collect()


## 9. Unseen-attack evaluation

The primary split answers *how well does this detect attacks like the ones it trained on*.
The day-disjoint split answers a harder and more realistic question: **how well does it detect
an attack family it has never seen?** Because each CIC attack ran on a specific day, holding out
Friday leaves Botnet, DDoS and PortScan entirely absent from training.

Expect a large drop. That drop is the honest answer to a question the benchmark literature
usually avoids, and it belongs in your evaluation chapter.

This cell evaluates the best model on attack types absent from its training data.
Training uses Monday to Wednesday while the Friday test day holds three unseen families.
A large drop is the expected and correct outcome rather than a failure.
Per class recall shows exactly which unseen families evade detection.

In [ ]:
# --- Day-disjoint unseen-attack evaluation ---------------------------------
UNSEEN = []
if "day" in df.columns:
    DAYS = {"train": ["Mon", "Tue", "Wed"], "val": ["Thu"], "test": ["Fri"]}
    di = {k: np.where(df["day"].isin(v).to_numpy())[0] for k, v in DAYS.items()}
    seen = set(df["y_fine"].iloc[di["train"]].unique())
    unseen_cls = sorted(set(df["y_fine"].iloc[di["test"]].unique()) - seen)
    print(f"train {len(di['train']):,} | val {len(di['val']):,} | test {len(di['test']):,}")
    print(f"Attack families in Friday test never seen in training: {unseen_cls}")

    Xd = {p: transform_block(preA, df, FSA[DS], CATS[DS], di[p])
          for p in ["train", "val", "test"]}
    yd = {p: y_all[di[p]] for p in ["train", "val", "test"]}

    for mname in ["RandomForest", "LogisticRegression", "MLP"]:
        if mname == "MLP":
            _, pv, pt, f_s, i_s, _ = run_mlp(Xd["train"], yd["train"],
                                             Xd["val"], yd["val"], Xd["test"])
        else:
            _, pv, pt, f_s, i_s, _ = MODEL_RUNNERS[mname](
                Xd["train"], yd["train"], Xd["val"], Xd["test"])
        r = evaluate(mname, "A_daydisjoint", yd["val"], pv, yd["test"], pt,
                     f_s, i_s, len(yd["test"]))
        UNSEEN.append(r)
        thr = r["fpr1_threshold"]
        fine = df["y_fine"].iloc[di["test"]].to_numpy()
        print(f"\n  {mname}: PR-AUC={r['pr_auc']:.4f}  DR@1%FPR={r['fpr1_recall']:.4f}")
        for c in unseen_cls:
            m_ = fine == c
            if m_.sum():
                print(f"      {c:14s} n={m_.sum():>7,}  "
                      f"recall@1%FPR={(pt[m_] >= thr).mean():.4f}   [UNSEEN]")
        gc.collect()

    U = pd.DataFrame(UNSEEN)
    U.to_csv(REPORTS / "unseen_attack_eval.csv", index=False)
    print("\nCompare against the primary split. The gap is the generalisation cost.")
    for m in U.model.unique():
        p = RES[(RES.model == m) & (RES.feature_set == "A")]
        if len(p):
            u = U[U.model == m].iloc[0]
            print(f"  {m:18s} PR-AUC {p.iloc[0].pr_auc:.4f} (seen) -> "
                  f"{u.pr_auc:.4f} (unseen)  "
                  f"{100*(u.pr_auc-p.iloc[0].pr_auc)/p.iloc[0].pr_auc:+.1f}%")
    M["results"]["unseen"] = U.to_dict("records")
    del Xd; gc.collect()


## 9b. Learning curve

Does the extra 1.03M training rows in Stage 2 actually buy anything over the 400k development
sample? This sweep answers that directly and justifies the two-stage protocol in the
methodology chapter. It also tells you whether collecting more data would help at all, which is
a question examiners ask and most dissertations cannot answer.

This cell measures how performance changes as the training set grows.
A light forest is used because only the shape of the curve matters here.
A flat tail means more data would not improve the detector any further.
The curve justifies the two stage protocol used throughout this notebook.

In [ ]:
# --- Learning curve ---------------------------------------------------------
LC_SIZES = [25_000, 50_000, 100_000, 200_000, 400_000, 800_000]
LC_SIZES = [s for s in LC_SIZES if s <= len(Y["train"])] + [len(Y["train"])]
LC_SIZES = sorted(set(LC_SIZES))
lc_rows = []
lc_rng = np.random.default_rng(SEED)

for n in LC_SIZES:
    idx = np.sort(lc_rng.choice(len(Y["train"]), size=n, replace=False))
    m = RandomForestClassifier(n_estimators=80, max_depth=22, min_samples_leaf=5,
                               max_features="sqrt", class_weight="balanced_subsample",
                               n_jobs=-1, random_state=SEED)
    t0 = time.time()
    m.fit(DATA["A"]["train"][idx], Y["train"][idx])
    fit_s = time.time() - t0
    pv = m.predict_proba(DATA["A"]["val"])[:, 1]
    pt = m.predict_proba(DATA["A"]["test"])[:, 1]
    thr = pick_thresholds(Y["val"], pv)
    lc_rows.append({"n_train": n,
                    "pr_auc": float(average_precision_score(Y["test"], pt)),
                    "roc_auc": float(roc_auc_score(Y["test"], pt)),
                    "dr_at_1fpr": at_threshold(Y["test"], pt, thr["fpr1"])["recall"],
                    "fit_seconds": round(fit_s, 1)})
    print(f"  n={n:>9,}  PR-AUC={lc_rows[-1]['pr_auc']:.5f}  "
          f"DR@1%FPR={lc_rows[-1]['dr_at_1fpr']:.5f}  ({fit_s:.0f}s)")
    del m; gc.collect()

LC = pd.DataFrame(lc_rows)
LC.to_csv(REPORTS / "learning_curve.csv", index=False)
gain = LC.pr_auc.iloc[-1] - LC[LC.n_train <= 400_000].pr_auc.max()
print(f"\nPR-AUC gained by going beyond 400k rows: {gain:+.5f}")
M["results"]["learning_curve"] = lc_rows


## 10. Figures for the dissertation

Every figure below is exported twice: **PNG at 300 dpi** for Word, and **PDF (vector)** for
LaTeX or for scaling without pixellation. A `figure_index.csv` is written at the end mapping
each file to a suggested caption and the chapter section it belongs in, so you can drop them
straight into the report without re-deriving what each one shows.

This cell sets up the figure export helper used by every plot that follows.
Each figure is written as a raster copy and as a vector copy for the report.
A caption and a chapter section are recorded alongside every file.
Consistent colours across figures make the results chapter easier to read.

In [ ]:
# --- Figure export helper ---------------------------------------------------
FIGURE_INDEX = []
COLORS = {"LogisticRegression": "#3b6ea5", "RandomForest": "#c1666b", "MLP": "#5a9367"}
FSCOL  = {"A": "#3b6ea5", "B": "#e0a458", "C": "#5a9367"}
plt.rcParams.update({"savefig.dpi": 300, "font.size": 9,
                     "axes.grid": True, "grid.alpha": 0.3})

def savefig(fig, name, caption, section):
    for ext in ("png", "pdf"):
        fig.savefig(FIGURES / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    FIGURE_INDEX.append({"file_png": f"{name}.png", "file_pdf": f"{name}.pdf",
                         "thesis_section": section, "suggested_caption": caption})
    plt.show(); plt.close(fig)
    print(f"  saved {name}.png / .pdf")

TESTFINE = df["y_fine"].iloc[sp["test"]].to_numpy()
PROBS = {}
for fs in ["A", "B", "C"]:
    for mn in MODEL_RUNNERS:
        p = REPORTS / f"probs_full_{mn}_{fs}.npy"
        if p.exists():
            PROBS[(mn, fs)] = np.load(p)
print(f"Loaded {len(PROBS)} full-scale probability vectors for plotting")


This cell draws precision recall and ROC curves for every model and feature set.
Precision recall is the informative view when the positive class is a minority.
The one percent false positive budget appears as a vertical reference line.
These two panels are the primary evidence figure for research question one.

In [ ]:
# --- F01: PR and ROC curves -------------------------------------------------
if PROBS:
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for j, fs in enumerate(["A", "B", "C"]):
        for mn in MODEL_RUNNERS:
            if (mn, fs) not in PROBS:
                continue
            pt = PROBS[(mn, fs)]
            pr, rc, _ = precision_recall_curve(Y["test"], pt)
            axes[0, j].plot(rc, pr, color=COLORS[mn], lw=1.7,
                            label=f"{mn} (AP={average_precision_score(Y['test'],pt):.3f})")
            fpr, tpr, _ = roc_curve(Y["test"], pt)
            axes[1, j].plot(fpr, tpr, color=COLORS[mn], lw=1.7,
                            label=f"{mn} (AUC={roc_auc_score(Y['test'],pt):.3f})")
        axes[0, j].axhline(Y["test"].mean(), ls=":", c="grey", lw=1,
                           label=f"random ({Y['test'].mean():.3f})")
        axes[0, j].set(title=f"Feature Set {fs} — Precision-Recall",
                       xlabel="recall", ylabel="precision", ylim=(0, 1.02))
        axes[1, j].axvline(CFG["fpr_budget"], ls="--", c="#c1666b", lw=1.2,
                           label=f"{100*CFG['fpr_budget']:.0f}% FPR budget")
        axes[1, j].set(title=f"Feature Set {fs} — ROC (log FPR)", xscale="log",
                       xlim=(1e-5, 1), ylim=(0, 1.02),
                       xlabel="false positive rate", ylabel="detection rate")
        axes[0, j].legend(fontsize=7, loc="lower left")
        axes[1, j].legend(fontsize=7, loc="lower right")
    fig.tight_layout()
    savefig(fig, "fig01_pr_roc_curves",
            "Precision-recall and ROC performance of the three models across Feature Sets "
            "A, B and C on the CIC-IDS2017 held-out test partition. The ROC axis is "
            "logarithmic to make the low-false-positive region legible.",
            "Results - RQ1 model comparison")


This cell plots detection rate at the fixed false positive budget per configuration.
Error bars carry the seed to seed spread measured during the development stage.
Bars whose error bars overlap are not meaningfully different from one another.
This is the clearest single visual answer to research question one.

In [ ]:
# --- F02: detection rate at fixed FPR budget --------------------------------
fig, ax = plt.subplots(figsize=(10, 5.5))
w, xs = 0.25, np.arange(3)
for i, mn in enumerate(MODEL_RUNNERS):
    mu, sd = [], []
    for fs in ["A", "B", "C"]:
        s = agg[(agg.model == mn) & (agg.feature_set == fs)]
        v = S2[(S2.model == mn) & (S2.feature_set == fs)] if len(S2) else pd.DataFrame()
        mu.append(float(v.fpr1_recall.iloc[0]) if len(v)
                  else (float(s.dr_at_1fpr_mean.iloc[0]) if len(s) else np.nan))
        sd.append(float(s.dr_at_1fpr_std.iloc[0]) if len(s) else 0.0)
    b = ax.bar(xs + (i - 1) * w, mu, w, yerr=sd, capsize=4, label=mn, color=COLORS[mn])
    for r, v in zip(b, mu):
        if not np.isnan(v):
            ax.text(r.get_x() + r.get_width()/2, v + 0.015, f"{v:.3f}",
                    ha="center", fontsize=8)
ax.set(xticks=xs, ylim=(0, 1.08),
       ylabel=f"detection rate at FPR <= {100*CFG['fpr_budget']:.0f}%",
       title="RQ1: detection at a fixed alert budget (error bars = seed spread)")
ax.set_xticklabels([f"Feature Set {f}" for f in ["A", "B", "C"]])
ax.legend()
fig.tight_layout()
savefig(fig, "fig02_detection_at_fixed_fpr",
        "Detection rate achieved at a one per cent false-positive budget. Error bars show "
        "the standard deviation across three random seeds measured at development scale. "
        "Overlapping error bars indicate differences that are not statistically meaningful.",
        "Results - RQ1 model comparison")


This cell shows how each metric degrades as the feature set is reduced.
Feature Set A is the full schema while B and C are progressively smaller.
Plotting several metrics together prevents a single number driving the conclusion.
This figure is the direct answer to the second experiment in the proposal.

In [ ]:
# --- F03: feature-set degradation profile -----------------------------------
RESF = S2 if len(S2) else S1[S1.seed == SEED]
metrics = [("pr_auc", "PR-AUC"), ("roc_auc", "ROC-AUC"),
           ("fpr1_recall", "DR @ 1% FPR"), ("f1_f1", "F1"),
           ("f1_mcc", "MCC"), ("f1_balanced_accuracy", "Balanced accuracy")]
fig, axes = plt.subplots(2, 3, figsize=(15, 7.5))
for ax, (col, lbl) in zip(axes.ravel(), metrics):
    for mn in MODEL_RUNNERS:
        ys = [float(RESF[(RESF.model == mn) & (RESF.feature_set == fs)][col].iloc[0])
              if len(RESF[(RESF.model == mn) & (RESF.feature_set == fs)]) else np.nan
              for fs in ["A", "B", "C"]]
        ax.plot([0, 1, 2], ys, "o-", color=COLORS[mn], lw=1.8, ms=6, label=mn)
    ax.set(xticks=[0, 1, 2], title=lbl, ylabel=lbl)
    ax.set_xticklabels([f"A\n({DATA['A']['train'].shape[1]}f)",
                        f"B\n({DATA['B']['train'].shape[1]}f)",
                        f"C\n({DATA['C']['train'].shape[1]}f)"])
axes[0, 0].legend(fontsize=8)
fig.suptitle("Experiment 2: performance cost of feature-set reduction", y=1.01)
fig.tight_layout()
savefig(fig, "fig03_feature_set_degradation",
        "Performance of each model across the three feature sets on six evaluation metrics. "
        "Feature Set A is the full cleaned schema, B the cross-dataset harmonised subset and "
        "C the reduced set selected on training data only. Feature counts are shown on the "
        "horizontal axis.",
        "Results - RQ1 feature-set comparison")


This cell shows recall for every individual attack family in the test set.
Aggregate scores hide the fact that rare attack types are often missed entirely.
Cells are annotated with the number of test examples behind each figure.
Reporting per class recall is what stops a headline number being misleading.

In [ ]:
# --- F04: per-attack-class recall heatmap -----------------------------------
if PROBS:
    classes = [c for c in pd.Series(TESTFINE).value_counts().index if c != "BENIGN"]
    rows, labels = [], []
    for mn in MODEL_RUNNERS:
        for fs in ["A", "B", "C"]:
            if (mn, fs) not in PROBS:
                continue
            r = RESF[(RESF.model == mn) & (RESF.feature_set == fs)]
            if not len(r):
                continue
            thr = float(r.fpr1_threshold.iloc[0])
            pt = PROBS[(mn, fs)]
            rows.append([float((pt[TESTFINE == c] >= thr).mean()) for c in classes])
            labels.append(f"{mn[:12]} / {fs}")
    Hm = np.array(rows)
    fig, ax = plt.subplots(figsize=(1.05 * len(classes) + 4, 0.42 * len(labels) + 3))
    im = ax.imshow(Hm, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(classes)))
    ax.set_xticklabels([f"{c}\n(n={int((TESTFINE==c).sum()):,})" for c in classes],
                       rotation=45, ha="right", fontsize=7)
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
    for (i, j), v in np.ndenumerate(Hm):
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                color="black" if 0.25 < v < 0.85 else "white")
    ax.set_title(f"Per-attack-family recall at FPR <= {100*CFG['fpr_budget']:.0f}%")
    ax.grid(False); fig.colorbar(im, ax=ax, shrink=0.8, label="recall")
    fig.tight_layout()
    savefig(fig, "fig04_per_class_recall",
            "Recall for each individual attack family at a one per cent false-positive "
            "budget. Test-set support is shown under each family name. Low-support families "
            "such as Heartbleed and Infiltration are detected far less reliably than the "
            "aggregate scores suggest.",
            "Results - RQ1 per-class analysis")


This cell traces detection rate across a wide range of alert budgets.
A security team with more analysts can tolerate a higher false positive rate.
Showing the whole curve is more useful than reporting one operating point.
This figure supports the deployment discussion in the evaluation chapter.

In [ ]:
# --- F05: detection rate versus alert budget --------------------------------
if PROBS:
    budgets = np.array([1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2, 1e-1])
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    for ax, fs in zip(axes, ["A", "B", "C"]):
        for mn in MODEL_RUNNERS:
            if (mn, fs) not in PROBS:
                continue
            fpr, tpr, _ = roc_curve(Y["test"], PROBS[(mn, fs)])
            ax.plot(budgets, np.interp(budgets, fpr, tpr), "o-",
                    color=COLORS[mn], lw=1.7, ms=5, label=mn)
        ax.axvline(CFG["fpr_budget"], ls="--", c="grey", lw=1)
        ax.set(xscale="log", ylim=(0, 1.02), title=f"Feature Set {fs}",
               xlabel="false positive rate (alert budget)", ylabel="detection rate")
        ax.legend(fontsize=7, loc="lower right")
    fig.suptitle("Detection rate as a function of the tolerated alert budget", y=1.02)
    fig.tight_layout()
    savefig(fig, "fig05_detection_vs_budget",
            "Detection rate achieved at alert budgets from 0.01 to 10 per cent false "
            "positives. The dashed line marks the one per cent operating point used for the "
            "headline results.",
            "Evaluation - operational trade-off")


This cell plots the confusion matrix of the best model at both thresholds.
The two panels make the detection versus false alarm trade off concrete.
Absolute counts are shown because analysts triage alerts not percentages.
This directly addresses the trade off named in research question one.

In [ ]:
# --- F06: confusion matrices at both thresholds -----------------------------
if len(RESF):
    b = RESF.sort_values("pr_auc", ascending=False).iloc[0]
    if (b.model, b.feature_set) in PROBS:
        pt = PROBS[(b.model, b.feature_set)]
        fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8))
        for ax, (tag, t) in zip(axes, [("F1-optimal threshold", b.f1_threshold),
                                       (f"FPR <= {100*CFG['fpr_budget']:.0f}% threshold",
                                        b.fpr1_threshold)]):
            cm = confusion_matrix(Y["test"], (pt >= t).astype(int), labels=[0, 1])
            ax.imshow(cm, cmap="Blues")
            for (r, c), v in np.ndenumerate(cm):
                ax.text(c, r, f"{v:,}\n({100*v/cm.sum():.2f}%)", ha="center",
                        va="center", fontsize=10,
                        color="white" if v > cm.max()/2 else "black")
            ax.set(xticks=[0, 1], yticks=[0, 1])
            ax.set_xticklabels(["predicted benign", "predicted attack"])
            ax.set_yticklabels(["true benign", "true attack"])
            rec = cm[1, 1] / max(cm[1].sum(), 1)
            ax.set_title(f"{tag}\nthr={t:.4f}   recall={rec:.4f}   "
                         f"false alerts={cm[0,1]:,}")
            ax.grid(False)
        fig.suptitle(f"{b.model} on Feature Set {b.feature_set} "
                     f"— test partition (n={len(Y['test']):,})", y=1.03)
        fig.tight_layout()
        savefig(fig, "fig06_confusion_matrices",
                "Confusion matrices for the best-performing configuration at the two "
                "validation-selected thresholds. The left panel maximises F1 and the right "
                "panel constrains the false-positive rate to one per cent.",
                "Results - RQ1 error analysis")


This cell shows how confidently each model separates benign traffic from attacks.
Heavy overlap between the two distributions means no threshold can work well.
A clean separation means the model is genuinely learning the distinction.
This diagnostic explains the shape of the curves in the earlier figures.

In [ ]:
# --- F07: score separation --------------------------------------------------
if PROBS:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
    for ax, mn in zip(axes, MODEL_RUNNERS):
        if (mn, "A") not in PROBS:
            continue
        pt = PROBS[(mn, "A")]
        ax.hist(pt[Y["test"] == 0], bins=60, alpha=.65, density=True,
                label="benign", color="#8bbf8b")
        ax.hist(pt[Y["test"] == 1], bins=60, alpha=.65, density=True,
                label="attack", color="#c1666b")
        r = RESF[(RESF.model == mn) & (RESF.feature_set == "A")]
        if len(r):
            ax.axvline(float(r.fpr1_threshold.iloc[0]), ls="--", c="black", lw=1.2,
                       label="1% FPR threshold")
        ax.set(yscale="log", title=f"{mn} — Feature Set A",
               xlabel="predicted attack probability", ylabel="density (log)")
        ax.legend(fontsize=7)
    fig.suptitle("Predicted-probability distributions by true class", y=1.03)
    fig.tight_layout()
    savefig(fig, "fig07_score_separation",
            "Distribution of predicted attack probabilities for benign and attack flows on "
            "the test partition. The dashed line marks the threshold satisfying a one per "
            "cent false-positive budget. Overlap between the distributions bounds the "
            "achievable detection rate.",
            "Results - RQ1 diagnostics")


This cell compares accuracy against the computational cost of each model.
Marker area encodes inference cost so all three axes are visible at once.
A model in the upper left corner is both accurate and cheap to train.
Computational cost is a secondary criterion the proposal requires reporting.

In [ ]:
# --- F08: accuracy versus cost frontier -------------------------------------
if len(RESF):
    fig, ax = plt.subplots(figsize=(9, 6))
    for mn in MODEL_RUNNERS:
        for fs in ["A", "B", "C"]:
            r = RESF[(RESF.model == mn) & (RESF.feature_set == fs)]
            if not len(r):
                continue
            r = r.iloc[0]
            ax.scatter(r.fit_seconds, r.pr_auc, s=40 + 900 * r.infer_us_per_flow /
                       max(RESF.infer_us_per_flow.max(), 1e-9),
                       color=COLORS[mn], alpha=.65,
                       edgecolor="black", linewidth=.6)
            ax.annotate(f"{mn[:2]}/{fs}", (r.fit_seconds, r.pr_auc),
                        textcoords="offset points", xytext=(7, 5), fontsize=8)
    for mn in MODEL_RUNNERS:
        ax.scatter([], [], color=COLORS[mn], label=mn, s=60)
    ax.set(xscale="log", xlabel="training time (seconds, log scale)", ylabel="test PR-AUC",
           title="Accuracy against computational cost\n(marker area is proportional to "
                 "inference microseconds per flow)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "fig08_accuracy_vs_cost",
            "Test PR-AUC against training time for all nine model and feature-set "
            "combinations. Marker area is proportional to inference cost per flow. "
            "Configurations toward the upper left offer the best accuracy per unit of "
            "computation.",
            "Evaluation - computational cost")


This cell displays the top ranked features and how the two rankings compare.
Mutual information is univariate while the forest ranking captures interactions.
Large disagreement usually marks a feature that only matters in combination.
This figure supports the feature selection argument in the methodology.

In [ ]:
# --- F09: feature importance ------------------------------------------------
top = RANK.head(25).iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(14, 7.5))
axes[0].barh(range(len(top)), top.mutual_info, color="#3b6ea5")
axes[0].set(yticks=range(len(top)), title="Mutual information (training data only)",
            xlabel="mutual information with the binary label")
axes[0].set_yticklabels([t[:34] for t in top.feature], fontsize=7)
axes[1].barh(range(len(top)), top.rf_importance, color="#c1666b")
axes[1].set(yticks=range(len(top)), title="Random Forest impurity importance",
            xlabel="importance")
axes[1].set_yticklabels([t[:34] for t in top.feature], fontsize=7)
for i, f in enumerate(top.feature):
    if f in FSC_NAMES:
        axes[0].text(-0.002, i, "*", va="center", ha="right",
                     fontsize=12, color="#5a9367")
fig.suptitle("Top 25 features by combined rank (asterisk marks membership of Feature Set C)",
             y=1.01)
fig.tight_layout()
savefig(fig, "fig09_feature_importance",
        "The twenty-five highest-ranked features under mutual information and Random Forest "
        "impurity importance, both computed on the training partition only. Features marked "
        "with an asterisk were retained in Feature Set C after correlation pruning.",
        "Results - feature analysis")


This cell shows how validation score varies with the size of the reduced feature set.
The chosen cut off is the point that maximised validation precision recall area.
A plateau means further features add cost without adding detection ability.
The test partition played no part in this selection at any stage.

In [ ]:
# --- F10: Feature Set C selection curve -------------------------------------
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(SW.k, SW.val_pr_auc, "o-", color="#3b6ea5", lw=1.9, ms=7)
ax.axvline(BEST_K, ls="--", c="#c1666b", lw=1.4, label=f"selected k = {BEST_K}")
for _, r in SW.iterrows():
    ax.annotate(f"{r.val_pr_auc:.4f}", (r.k, r.val_pr_auc),
                textcoords="offset points", xytext=(0, 9), ha="center", fontsize=7)
ax.set(xlabel="number of features retained (k)", ylabel="validation PR-AUC",
       title=f"Feature Set C size selected on validation\n"
             f"(after pruning {len(M['results']['corr_pruned'])} near-duplicate columns "
             f"at |r| > {CFG['corr_prune_threshold']})")
ax.legend()
fig.tight_layout()
savefig(fig, "fig10_feature_set_c_selection",
        "Validation PR-AUC as a function of the number of features retained in Feature Set "
        "C. The cut-off was selected on validation data only; the test partition took no "
        "part in this decision.",
        "Methodology - feature selection")


This cell shows the run to run variability of every configuration.
Three seeds were used at development scale to estimate this spread.
A performance gap smaller than the spread is not evidence of a better model.
Reporting variability rather than a single number is a proposal requirement.

In [ ]:
# --- F11: seed variability ---------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (col, lbl) in zip(axes, [("pr_auc", "PR-AUC"),
                                 ("fpr1_recall", "detection rate at 1% FPR")]):
    groups, labels, cols = [], [], []
    for fs in ["A", "B", "C"]:
        for mn in MODEL_RUNNERS:
            g = S1[(S1.model == mn) & (S1.feature_set == fs)][col].to_numpy()
            if len(g):
                groups.append(g); labels.append(f"{mn[:2]}/{fs}"); cols.append(COLORS[mn])
    bp = ax.boxplot(groups, labels=labels, patch_artist=True, widths=.55)
    for patch, c in zip(bp["boxes"], cols):
        patch.set_facecolor(c); patch.set_alpha(.65)
    ax.set(title=f"{lbl} across {len(CFG['seeds_stage1'])} seeds (development scale)",
           ylabel=lbl)
    ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
savefig(fig, "fig11_seed_variability",
        "Distribution of test performance across three random seeds at development scale "
        "for each model and feature-set combination. The spread defines the threshold below "
        "which differences between configurations cannot be considered meaningful.",
        "Results - reliability of comparisons")


This cell visualises how much UNSW performance depends on three artefact features.
The traffic generator emitted a near fixed time to live value for attack traffic.
Removing those columns shows how much apparent skill was never real.
This is the strongest methodological finding in the whole notebook.

In [ ]:
# --- F12: UNSW artefact ablation --------------------------------------------
if len(ABL):
    A = pd.DataFrame(ABL)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.6))
    for ax, (col, lbl) in zip(axes, [("pr_auc", "PR-AUC"), ("roc_auc", "ROC-AUC"),
                                     ("fpr1_recall", "DR @ 1% FPR")]):
        mods = A.model.unique(); xs = np.arange(len(mods))
        w_ = [float(A[(A.model == m) & (A.feature_set == "with_artefacts")][col].iloc[0])
              for m in mods]
        o_ = [float(A[(A.model == m) & (A.feature_set == "without_artefacts")][col].iloc[0])
              for m in mods]
        ax.bar(xs - .2, w_, .4, label="with sttl / dttl / ct_state_ttl", color="#c1666b")
        ax.bar(xs + .2, o_, .4, label="artefacts removed", color="#3b6ea5")
        for x, (a_, b_) in enumerate(zip(w_, o_)):
            ax.text(x - .2, a_ + .012, f"{a_:.3f}", ha="center", fontsize=7)
            ax.text(x + .2, b_ + .012, f"{b_:.3f}", ha="center", fontsize=7)
            ax.annotate(f"{100*(b_-a_)/max(a_,1e-9):+.1f}%", (x, max(a_, b_) + .07),
                        ha="center", fontsize=8, color="#8a3b3b")
        ax.set(xticks=xs, ylim=(0, 1.15), title=lbl, ylabel=lbl)
        ax.set_xticklabels([m[:14] for m in mods], fontsize=8)
    axes[0].legend(fontsize=7)
    fig.suptitle("UNSW-NB15: performance attributable to three testbed artefact features",
                 y=1.03)
    fig.tight_layout()
    savefig(fig, "fig12_unsw_artefact_ablation",
            "Effect of removing the three time-to-live derived features from UNSW-NB15. "
            "The generator emitted a near-constant TTL for attack traffic, so these columns "
            "separate the classes almost perfectly within the dataset while carrying no "
            "transferable signal.",
            "Evaluation - dataset validity")


This cell compares detection of familiar attacks against completely unseen ones.
Training used Monday to Wednesday while Friday held three unseen attack families.
A large drop is the expected and correct outcome rather than a failure.
This figure answers a question most benchmark studies avoid entirely.

In [ ]:
# --- F13: unseen-attack generalisation --------------------------------------
if len(UNSEEN):
    U = pd.DataFrame(UNSEEN)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    mods = list(U.model); xs = np.arange(len(mods))
    for ax, (col, lbl) in zip(axes, [("pr_auc", "PR-AUC"),
                                     ("fpr1_recall", "DR @ 1% FPR")]):
        seen = [float(RESF[(RESF.model == m) & (RESF.feature_set == "A")][col].iloc[0])
                if len(RESF[(RESF.model == m) & (RESF.feature_set == "A")]) else np.nan
                for m in mods]
        uns = [float(U[U.model == m][col].iloc[0]) for m in mods]
        ax.bar(xs - .2, seen, .4, label="stratified split (attack types seen)",
               color="#3b6ea5")
        ax.bar(xs + .2, uns, .4, label="day-disjoint split (unseen attack types)",
               color="#e0a458")
        for x, (a_, b_) in enumerate(zip(seen, uns)):
            ax.text(x - .2, a_ + .012, f"{a_:.3f}", ha="center", fontsize=7)
            ax.text(x + .2, b_ + .012, f"{b_:.3f}", ha="center", fontsize=7)
        ax.set(xticks=xs, ylim=(0, 1.12), title=lbl, ylabel=lbl)
        ax.set_xticklabels([m[:16] for m in mods], fontsize=8)
    axes[0].legend(fontsize=7)
    fig.suptitle("Generalisation to attack families absent from training "
                 f"({', '.join(unseen_cls)})", y=1.03)
    fig.tight_layout()
    savefig(fig, "fig13_unseen_attack_generalisation",
            "Comparison of detection performance on the stratified split, where all attack "
            "families appear in training, against the day-disjoint split, where Botnet, "
            "DDoS and PortScan traffic is entirely absent from the training partition.",
            "Evaluation - zero-day robustness")


This cell shows how the neural model improved over its training epochs.
Early stopping watched validation precision recall area rather than loss.
The marked epoch is the point whose weights were restored and evaluated.
This evidences the training protocol described in the methodology.

In [ ]:
# --- F14: MLP training curves -----------------------------------------------
if MLP_HISTORY:
    fig, ax = plt.subplots(figsize=(9, 5))
    for fs, hist in MLP_HISTORY.items():
        h = pd.DataFrame(hist)
        ax.plot(h.epoch, h.val_pr_auc, "-", lw=1.8, color=FSCOL[fs],
                label=f"Feature Set {fs}")
        bi = int(h.val_pr_auc.idxmax())
        ax.scatter([h.epoch.iloc[bi]], [h.val_pr_auc.iloc[bi]], s=70,
                   color=FSCOL[fs], edgecolor="black", zorder=5)
        ax.annotate(f"best {h.val_pr_auc.iloc[bi]:.4f}",
                    (h.epoch.iloc[bi], h.val_pr_auc.iloc[bi]),
                    textcoords="offset points", xytext=(6, -12), fontsize=7)
    ax.set(xlabel="epoch", ylabel="validation PR-AUC",
           title="MLP training progress with early stopping on validation PR-AUC")
    ax.legend()
    fig.tight_layout()
    savefig(fig, "fig14_mlp_training_curves",
            "Validation PR-AUC per epoch for the multilayer perceptron on each feature set. "
            "Training stopped when the metric failed to improve for seven consecutive "
            "epochs and the best-scoring weights were restored. Markers indicate the "
            "restored epoch.",
            "Methodology - neural model training")


This cell shows how detection improves as the training set grows.
A flat right hand tail means further data collection would not help.
Training time is plotted alongside so the cost of scaling is visible.
This justifies the two stage protocol used throughout the notebook.

In [ ]:
# --- F15: learning curve -----------------------------------------------------
fig, ax = plt.subplots(figsize=(9.5, 5.5))
ax.plot(LC.n_train, LC.pr_auc, "o-", color="#3b6ea5", lw=1.9, ms=6, label="PR-AUC")
ax.plot(LC.n_train, LC.dr_at_1fpr, "s--", color="#c1666b", lw=1.7, ms=5,
        label="DR @ 1% FPR")
ax.set(xscale="log", xlabel="training rows (log scale)", ylabel="test performance",
       title="Learning curve — does more training data still help?")
ax2 = ax.twinx()
ax2.plot(LC.n_train, LC.fit_seconds, "^:", color="grey", lw=1.3, ms=5,
         label="training time")
ax2.set_ylabel("training time (s)"); ax2.grid(False)
h1, l1 = ax.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="lower right")
fig.tight_layout()
savefig(fig, "fig15_learning_curve",
        "Test performance and training cost as a function of training-set size for a Random "
        "Forest on Feature Set A. Flattening of the curve indicates the point beyond which "
        "additional training data yields negligible improvement.",
        "Evaluation - data sufficiency")


This cell checks whether the predicted probabilities can be read as real probabilities.
A well calibrated model sits close to the diagonal reference line.
Tree ensembles are usually over confident while linear models are closer.
Calibration matters if an analyst is to triage alerts by predicted risk.

In [ ]:
# --- F16: calibration --------------------------------------------------------
if PROBS:
    from sklearn.calibration import calibration_curve
    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    ax.plot([0, 1], [0, 1], "k:", lw=1.2, label="perfectly calibrated")
    for mn in MODEL_RUNNERS:
        if (mn, "A") not in PROBS:
            continue
        pt = PROBS[(mn, "A")]
        frac, mean_p = calibration_curve(Y["test"], pt, n_bins=15, strategy="quantile")
        brier = float(np.mean((pt - Y["test"]) ** 2))
        ax.plot(mean_p, frac, "o-", color=COLORS[mn], lw=1.7, ms=5,
                label=f"{mn} (Brier={brier:.4f})")
    ax.set(xlabel="mean predicted probability", ylabel="observed attack fraction",
           title="Calibration of predicted attack probabilities (Feature Set A)")
    ax.legend(fontsize=8)
    fig.tight_layout()
    savefig(fig, "fig16_calibration",
            "Reliability diagram comparing predicted attack probability against observed "
            "attack frequency on the test partition, with Brier scores. Deviation from the "
            "diagonal indicates miscalibration, which matters when alerts are triaged by "
            "predicted risk.",
            "Evaluation - probability quality")


This cell writes an index of every figure with a caption and a chapter section.
The index makes assembling the results chapter a matter of copying rows.
Each figure exists as a raster file and as a vector file for the report.
Figures from the two earlier notebooks are catalogued here as well.

In [ ]:
# --- Figure index for the dissertation --------------------------------------
prev = [
    ("nb01", "fig01_class_distributions", "Data understanding",
     "Class distribution of the three benchmark datasets on a logarithmic scale."),
    ("nb01", "fig02_data_quality", "Data understanding",
     "Missing values, infinite values and duplicate-row rates per dataset."),
    ("nb01", "fig03_leakage_screen", "Methodology - leakage control",
     "Single-feature separability screen identifying candidate artefact features."),
    ("nb01", "fig04_covariate_shift", "Data understanding",
     "Distribution comparison of shared features between CIC-IDS2017 and UNSW-NB15."),
    ("nb01", "fig05_correlation", "Data understanding",
     "Pearson correlation structure of the CIC-IDS2017 numeric features."),
    ("nb02", "fig01_transform_effect", "Methodology - preprocessing",
     "Effect of the winsorise, signed-log and robust-scale chain on a heavy-tailed feature."),
    ("nb02", "fig02_residual_shift", "Methodology - harmonisation",
     "Residual distributional distance between datasets after unit harmonisation."),
]
IDX = pd.DataFrame(
    [{"notebook": "nb03", **f} for f in FIGURE_INDEX] +
    [{"notebook": nbk, "file_png": f"{n}.png", "file_pdf": f"{n}.pdf",
      "thesis_section": s, "suggested_caption": c} for nbk, n, s, c in prev])
IDX.insert(0, "figure_no", [f"Figure {i+1}" for i in range(len(IDX))])
IDX.to_csv(PROJECT / "reports" / "figure_index.csv", index=False)
display(IDX[["figure_no", "notebook", "file_png", "thesis_section"]])

print(f"\n{len(FIGURE_INDEX)} figures produced by this notebook, "
      f"{len(IDX)} catalogued in total.")
print(f"Index written to {PROJECT/'reports'/'figure_index.csv'}")
print(f"Figures directory: {FIGURES}")
print("\nEach figure exists twice: .png at 300 dpi for Word and .pdf vector for LaTeX.")


## 11. Save and hand-off

This cell writes the manifest and prints the summary for the next notebook.
Every number in the block was measured rather than assumed.
The printed block is short enough to paste directly into a conversation.
Notebook 04 begins from the artefacts listed at the end.

In [ ]:
# --- Manifest and hand-off --------------------------------------------------
M["run_finished_utc"] = datetime.now(timezone.utc).isoformat(timespec="seconds")
(REPORTS / "run_manifest_nb03.json").write_text(json.dumps(M, indent=2, default=str))

print("=" * 78)
print("NOTEBOOK 03 SUMMARY  — paste this back into the chat")
print("=" * 78)
print(f"\n[data]  CIC {len(df):,} rows  attack {100*df['y'].mean():.3f}%  "
      f"train/val/test {len(sp['train']):,}/{len(sp['val']):,}/{len(sp['test']):,}")
print(f"[feature sets]  A={DATA['A']['train'].shape[1]}  "
      f"B={DATA['B']['train'].shape[1]}  C={DATA['C']['train'].shape[1]} (k={BEST_K})")
print(f"[corr prune] removed {len(M['results']['corr_pruned'])} near-duplicate columns")

print("\n[RQ1 headline — full scale]")
for _, r in RES.sort_values("pr_auc", ascending=False).iterrows():
    print(f"  {r.feature_set} | {r.model:18s} PR-AUC={r.pr_auc:.4f}  "
          f"ROC-AUC={r.roc_auc:.4f}  DR@1%FPR={r.fpr1_recall:.4f}  "
          f"F1={r.f1_f1:.4f}  FPR@F1={r.f1_fpr:.5f}  "
          f"fit={r.fit_seconds:.0f}s  {r.infer_us_per_flow:.1f}us/flow")

print("\n[seed spread from stage 1]")
print(f"  max PR-AUC std across configurations: {agg.pr_auc_std.max():.5f}")
print(f"  max DR@1%FPR std: {agg.dr_at_1fpr_std.max():.5f}")

if len(ABL):
    print("\n[UNSW artefact ablation]")
    for m in pd.DataFrame(ABL).model.unique():
        A = pd.DataFrame(ABL)
        w  = A[(A.model == m) & (A.feature_set == "with_artefacts")].iloc[0]
        wo = A[(A.model == m) & (A.feature_set == "without_artefacts")].iloc[0]
        print(f"  {m:18s} PR-AUC {w.pr_auc:.4f} -> {wo.pr_auc:.4f}  "
              f"ROC-AUC {w.roc_auc:.4f} -> {wo.roc_auc:.4f}")

if len(UNSEEN):
    print("\n[unseen-attack day-disjoint]")
    for r in UNSEEN:
        print(f"  {r['model']:18s} PR-AUC={r['pr_auc']:.4f}  "
              f"DR@1%FPR={r['fpr1_recall']:.4f}")

print("\n" + "=" * 78)
print("\nArtefacts:")
for p in sorted(REPORTS.glob("*")):
    print(f"  {p.name:40s} {p.stat().st_size/1024:9.1f} KB")


---

## What to send back

The printed summary block. If you can also attach `fig01_pr_roc.png` and
`rq1_headline.csv`, I can read the shape of the curves before writing Notebook 04.

## What Notebook 04 does

Experiment 4 for RQ2 — the cross-dataset transfer:

- Train on CIC-IDS2017 with Feature Set B, evaluate frozen on UNSW-NB15, and the reverse
- **Always apply the source-fitted preprocessor to the target data.** The CIC and UNSW Set B
  preprocessors produced 32 and 33 columns respectively because their one-hot service
  vocabularies differ; using the target's own preprocessor would silently change the input space
- Threshold transfer analysis: a threshold chosen on the source validation set will be wrong on
  the target, and quantifying by how much is a genuine contribution
- Attribution of the gap: covariate shift (measured in NB02), label-definition shift, and the
  Feature Set B projection collision rate of 11.4% / 23.1%

## Things to be ready to defend

1. **Why detection rate at 1% FPR rather than accuracy.** At 15% attack prevalence a
   trivial always-benign classifier scores 84.9% accuracy. The proposal rules out accuracy-only
   selection for exactly this reason.
2. **Why thresholds come from validation.** Tuning a cut-off on the test set and then reporting
   test performance at that cut-off is optimistically biased, and it is extremely common in the
   NIDS literature.
3. **Why feature selection was recomputed here.** The Notebook 02 ranking used a training
   partition that no longer exists; some of those rows are now in test.
4. **Why the seed spread is reported.** A 0.002 PR-AUC gap between two models means nothing if
   the same model varies by 0.004 across seeds.